# Tree operations

This notebook covers operations on trees encoded as Phylo2Vec vectors (and matrices):
* Finding common ancestors and node depths
* Removing and adding leaves
* Rerooting
* Ordering trees with Queue Shuffle
* Processing Newick strings and taxon labels

To get familiar with the Phylo2Vec format first, see [Getting started](getting-started.ipynb) and [Core concepts](core-concepts.ipynb).

## Imports

In [ ]:
import numpy as np

from ete4 import Tree

import phylo2vec as p2v


def plot_tree(newick):
    print(Tree(newick))

We use the following tree with 7 leaves throughout:

In [ ]:
v_fixed = np.array([0, 2, 2, 5, 4, 1])

plot_tree(p2v.to_newick(v_fixed))

   ╭─┬╴0
 ╭─┤ ╰─┬╴1
 │ │   ╰╴6
─┤ ╰─┬╴4
 │   ╰╴5
 ╰─┬╴2
   ╰╴3


## 1. Common ancestors and node depths

Phylo2Vec provides functions for tree traversal, such as finding the most recent common ancestor (MRCA) between two nodes and computing node depths.

Use `get_common_ancestor` to find the MRCA between two nodes (similar to ape's `getMRCA` in R or ETE's `get_common_ancestor` in Python), and `get_node_depth` / `get_node_depths` to compute depths.

- For **vectors**: topological depth is returned (all branch lengths = 1)
- For **matrices**: actual branch lengths are used

**Tip:** Use `get_node_depths` (plural) to compute all node depths at once, which is more efficient than calling `get_node_depth` repeatedly.

In [ ]:
from phylo2vec.utils.vector import get_common_ancestor, get_node_depth, get_node_depths

# Using v_fixed from earlier: [0, 2, 2, 5, 4, 1]
# Tree structure (7 leaves, so internal nodes are 7-12, root is 12):
#    ╭─┬╴0
#  ╭─┤ ╰─┬╴1
#  │ │   ╰╴6
# ─┤ ╰─┬╴4
#  │   ╰╴5
#  ╰─┬╴2
#    ╰╴3

# Find MRCA of leaves 1 and 6 (they form a cherry, so MRCA is node 7)
mrca_1_6 = get_common_ancestor(v_fixed, 1, 6)
print(f"MRCA of leaves 1 and 6: node {mrca_1_6}")
assert mrca_1_6 == 7  # node 7 is the parent of leaves 1 and 6

# Get the depth of this MRCA (topological, since v_fixed is a vector)
mrca_depth = get_node_depth(v_fixed, mrca_1_6)
print(f"Depth of MRCA (node {mrca_1_6}): {mrca_depth}")
assert mrca_depth == 3  # root(12) → 11 → 10 → 7, so depth is 3

# Find MRCA of leaves 2 and 5 (MRCA is root, node 12)
mrca_2_5 = get_common_ancestor(v_fixed, 2, 5)
mrca_2_5_depth = get_node_depth(v_fixed, mrca_2_5)
print(f"MRCA of leaves 2 and 5: node {mrca_2_5}, depth: {mrca_2_5_depth}")
assert mrca_2_5 == 12  # root node
assert mrca_2_5_depth == 0  # root has depth 0

# Get all node depths at once
all_depths = get_node_depths(v_fixed)
print(f"All node depths: {all_depths}")
assert all_depths[12] == 0  # root depth is 0

MRCA of leaves 1 and 6: node 7
Depth of MRCA (node 7): 3.0
MRCA of leaves 2 and 5: node 12, depth: 0.0
All node depths: [3. 4. 2. 2. 3. 3. 4. 3. 2. 1. 2. 1. 0.]


With a matrix, node depths are computed from the branch lengths:

In [ ]:
m = p2v.from_newick("((0:0.1,1:0.2)3:0.3,2:0.4)4;")

print(f"matrix:\n{repr(m)}")
print(f"Node depths: {get_node_depths(m)}")

matrix:
array([[0. , 0.1, 0.2],
       [2. , 0.3, 0.4]])
Node depths: [0.4 0.5 0.4 0.3 0. ]


## 2. Removing and adding leaves

One might want to prune or add nodes in an existing tree (a common example is the subtree-prune-and-regraft operation).

This is not a trivial operation as we need to re-compute the vector (as the number of leaves in the tree will have changed)

Use `remove_leaf` to remove a leaf. Leaves with a higher label are relabelled (decremented by one).

In [ ]:
from phylo2vec.utils.vector import remove_leaf

v7 = p2v.sample_vector(7)

leaf = 3

# sister_leaf: sister node of `leaf` in `v7`
v6, sister_leaf = remove_leaf(v7, leaf=leaf)

In [ ]:
print("Before:")
plot_tree(p2v.to_newick(v7))

print(f"After removing leaf {leaf}:")
plot_tree(p2v.to_newick(v6))

Before:
   ╭╴0
 ╭─┤ ╭─┬╴1
 │ ╰─┤ ╰╴2
─┤   │ ╭─┬╴3
 │   ╰─┤ ╰╴4
 │     ╰╴6
 ╰╴5
After removing leaf 3:
   ╭╴0
 ╭─┤ ╭─┬╴1
 │ ╰─┤ ╰╴2
─┤   ╰─┬╴3
 │     ╰╴5
 ╰╴4


Use `add_leaf` to add the leaf back next to its sister:

In [ ]:
from phylo2vec.utils.vector import add_leaf

# due to re-labelling in remove_leaf, we have to decrement sister_leaf
if sister_leaf >= leaf:
    sister_leaf -= 1

v_add = add_leaf(v6, leaf=3, pos=sister_leaf)

np.array_equal(v_add, v7)

True

## 3. Rerooting

Use `reroot` to reroot a tree at a given node (a leaf or an internal node, except the current root): the new root is placed on the branch leading to that node. `reroot_at_random` picks the node at random.

In [ ]:
from phylo2vec.utils.vector import reroot

v_rerooted = reroot(v_fixed, node=4)

print("Before:")
plot_tree(p2v.to_newick(v_fixed))

print("After rerooting at leaf 4:")
plot_tree(p2v.to_newick(v_rerooted))

Before:
   ╭─┬╴0
 ╭─┤ ╰─┬╴1
 │ │   ╰╴6
─┤ ╰─┬╴4
 │   ╰╴5
 ╰─┬╴2
   ╰╴3
After rerooting at leaf 4:
     ╭─┬╴0
   ╭─┤ ╰─┬╴1
   │ │   ╰╴6
 ╭─┤ ╰─┬╴2
─┤ │   ╰╴3
 │ ╰╴5
 ╰╴4


## 4. Ordering a tree with Queue Shuffle

A vector is *ordered* if `v[i] <= i` for all `i` (see [Core concepts](core-concepts.ipynb)). Ordered vectors are faster to convert and are used by some optimisation methods.

Use `queue_shuffle` to relabel the leaves of any tree so that its vector is ordered, without changing the topology. It returns the ordered vector and a mapping: leaf `i` in the ordered vector is leaf `mapping[i]` in the original vector.

In [ ]:
from phylo2vec.utils.vector import queue_shuffle

v_ordered, mapping = queue_shuffle(v_fixed)

print(f"v (original): {repr(v_fixed)}")
print(f"v (ordered):  {repr(v_ordered)}")
print(f"mapping: {mapping}")

v (original): array([0, 2, 2, 5, 4, 1])
v (ordered):  array([0, 0, 1, 1, 3, 3])
mapping: [3, 5, 2, 6, 4, 0, 1]


Renaming each leaf `i` of the ordered tree to `mapping[i]` gives back the original tree:

In [ ]:
from phylo2vec.utils.newick import apply_label_mapping, remove_parent_labels

newick_ordered = remove_parent_labels(p2v.to_newick(v_ordered))

newick_renamed = apply_label_mapping(
    newick_ordered, {i: str(leaf) for i, leaf in enumerate(mapping)}
)

assert np.array_equal(p2v.from_newick(newick_renamed), v_fixed)

## 5. Processing Newick strings

### 5.1. Finding the number of leaves

In [ ]:
from phylo2vec.utils.newick import find_num_leaves

find_num_leaves(p2v.to_newick(v7))

7

### 5.2. Removing branch lengths and internal labels

In [ ]:
from phylo2vec.utils.newick import remove_branch_lengths

newick_with_bls = "((0:0.1,1:0.2)3:0.3,2:0.4)4;"

print(remove_branch_lengths(newick_with_bls))
print(remove_parent_labels(remove_branch_lengths(newick_with_bls)))

((0,1)3,2)4;
((0,1),2);


### 5.3. Taxon labels

* Newick strings usually do not contain integers but real-life taxa (e.g., animal species, languages...). So it is important to provide another layer of conversion, where we can take in a Newick with string taxa, and convert it to a Newick with integer taxa, with a unique integer → taxon mapping.

In [ ]:
n_leaves = 8

t = Tree()
t.populate(n_leaves)
nw_str = t.write(parser=9)

print(nw_str)

print(t)

(((b,f),(((d,c),h),(g,a))),e);
   ╭─┬╴b
   │ ╰╴f
 ╭─┤   ╭─┬╴d
 │ │ ╭─┤ ╰╴c
─┤ ╰─┤ ╰╴h
 │   ╰─┬╴g
 │     ╰╴a
 ╰╴e


In [ ]:
from pprint import pprint
from phylo2vec.utils.newick import create_label_mapping

nw_int, label_mapping = create_label_mapping(nw_str)

plot_tree(nw_int)

pprint(label_mapping)

   ╭─┬╴0
   │ ╰╴1
 ╭─┤   ╭─┬╴2
 │ │ ╭─┤ ╰╴3
─┤ ╰─┤ ╰╴4
 │   ╰─┬╴5
 │     ╰╴6
 ╰╴7
{0: 'b', 1: 'f', 2: 'd', 3: 'c', 4: 'h', 5: 'g', 6: 'a', 7: 'e'}


* The reverse operation is ```apply_label_mapping```

In [ ]:
from phylo2vec.utils.newick import apply_label_mapping

new_nw_str = apply_label_mapping(nw_int, label_mapping)

new_nw_str == nw_str

True